# Can the first order predict a 90-day return?

## 1 · Build one row per eligible customer

First observed order · Full 90-day follow-up

Run `sql/first_order_features.sql` in Snowflake and export its result to `data/processed/model_data.csv`, or run the local reproduction command in the README. SQL is kept as a separate file so this notebook uses an ordinary Python kernel.


## 2 · Use what the first order tells us

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
model_data = pd.read_csv(ROOT / "data/processed/model_data.csv")
model_data = model_data.sort_values(["CUSTOMERID", "INVOICENO"]).reset_index(drop=True)
model_data["RETURNED_WITHIN_90"].value_counts()


In [ ]:
feature_columns = ["ORDER_VALUE", "TOTAL_QUANTITY", "DISTINCT_PRODUCTS"]

## 3 · Hold out customers before training

Stratified 80/20 split · Seed 42

In [ ]:
from sklearn.model_selection import train_test_split

X = model_data[feature_columns]
y = model_data["RETURNED_WITHIN_90"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

## 4 · Fit and predict

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

model.fit(X_train, y_train);

In [ ]:
predictions = model.predict(X_test)
print(predictions[:10])

## 5 · Beat a simple baseline?

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
probabilities = model.predict_proba(X_test)[:, 1]

print("Model accuracy:", accuracy_score(y_test, predictions))
print("Baseline accuracy:", baseline.score(X_test, y_test))
print("ROC AUC:", roc_auc_score(y_test, probabilities))

## 6 · See the missed returns

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test, predictions,
    labels=[0, 1],
    display_labels=["No return in 90 days", "Return in 90 days"],
    cmap="Blues",
    values_format="d",
    colorbar=False
)
plt.title("90-day return prediction: test customers")
plt.tight_layout()
plt.show()

## 7 · Separate signal from chance

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(6, 4))
RocCurveDisplay.from_predictions(
    y_test, probabilities,
    name="Logistic regression",
    ax=ax
)
ax.plot([0, 1], [0, 1], "--", color="gray", label="Chance (AUC = 0.50)")
ax.set_title("Can the model distinguish returning customers?")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Interpretation
This is a random historical holdout, not a future-period validation. First observed purchase is not necessarily a new customer. The original showcase had AUC 0.546, near chance. See the current local rerun in `reports/metrics.json`; different row ordering can change holdout membership even with seed 42. Do not tune against this test set.
